# 🤖 Agentic Recruiting Agent — Project 4/4

Analyzes a job description + a folder of resumes and produces a ranked,
evidence-backed shortlist for **human review**.

- Runs 100% in Google Colab
- Uses only the **free-tier Gemini API** (`google-genai`)
- No LangChain / LangGraph / CrewAI / AutoGen
- No Docker, no backend server, nothing to install locally
- ~4 Gemini calls per run, all batched, to stay inside free-tier rate limits



## 1. Install dependencies

In [1]:
!pip install -q -U google-genai pypdf


## 2. Imports

In [2]:
import os
import json
import sys

print("Python:", sys.version)


Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


## 3. Load the Gemini API key from Colab Secrets

Click the 🔑 icon in the left sidebar → **Secrets** → add a secret named
`GEMINI_API_KEY` → toggle notebook access on. The key is never printed
or hard-coded here.


In [3]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY secret not found. Add it via the 🔑 Secrets panel "
        "on the left and turn on notebook access, then re-run this cell."
    )

print("✓ Gemini API key loaded from Colab Secrets (not printed).")


✓ Gemini API key loaded from Colab Secrets (not printed).


## 4. Configuration

This writes `config.py` to the Colab filesystem. Every tunable (scoring
weights, thresholds, model name, free-tier limits) lives here — change
values in this cell and re-run if you want different behavior.


In [4]:
%%writefile config.py
"""
config.py
---------
All the knobs for the recruiting agent live here so nothing is buried
inside logic files. Change these instead of hunting through the code.
"""

import os

# ---------------------------------------------------------------------
# Gemini model
# ---------------------------------------------------------------------
# Keep this overridable via env var so swapping models (e.g. if your
# account only has access to gemini-2.0-flash, or a newer flash model
# shows up) doesn't require touching any other file.
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")

# Free tier accounts are rate limited pretty aggressively (this project
# was built against a key limited to ~20 requests/minute). The agent is
# deliberately designed to use only 3-4 calls per run, but we still add
# a small delay + retry wrapper around every call so a transient 429
# doesn't blow up the whole run.
GEMINI_MAX_RETRIES = 3
GEMINI_RETRY_BASE_DELAY_SECONDS = 4
GEMINI_REQUEST_TIMEOUT_SECONDS = 60

# ---------------------------------------------------------------------
# Agent loop
# ---------------------------------------------------------------------
MAX_ITERATIONS = 6  # hard ceiling, agent must never loop past this

# ---------------------------------------------------------------------
# Free tier protection
# ---------------------------------------------------------------------
MAX_CANDIDATES = 10          # don't process more than this in one run
MAX_RESUME_CHARS = 12000     # truncate long resumes before sending to Gemini

# ---------------------------------------------------------------------
# Scoring weights (must sum to 1.0 - validated in scoring.py)
# ---------------------------------------------------------------------
SCORING_WEIGHTS = {
    "mandatory_skills": 0.50,
    "preferred_skills": 0.15,
    "experience": 0.20,
    "projects": 0.10,
    "evidence": 0.05,
}

# ---------------------------------------------------------------------
# Shortlist
# ---------------------------------------------------------------------
SHORTLIST_SIZE = 3
MIN_MANDATORY_COVERAGE = 0.70  # candidate needs >=70% mandatory skills matched

# ---------------------------------------------------------------------
# Fairness / privacy
# ---------------------------------------------------------------------
# These fields must never be used for scoring, even if they somehow show
# up in an extracted candidate profile. tools.py strips them before the
# profile is passed anywhere near the scoring engine.
PROHIBITED_FIELDS = {
    "age",
    "gender",
    "sex",
    "religion",
    "race",
    "ethnicity",
    "health",
    "disability",
    "marital_status",
    "political_affiliation",
    "sexual_orientation",
    "nationality",
    "photo",
    "photograph",
}

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DATA_DIR = "data"
RESUME_DIR = os.path.join(DATA_DIR, "resumes")
OUTPUT_DIR = "outputs"
JOB_DESCRIPTION_PATH = os.path.join(DATA_DIR, "job_description.txt")

DISCLAIMER = (
    "This system provides decision support only.\n"
    "Final hiring decisions must be made by qualified human reviewers."
)


Overwriting config.py


## 5. Gemini wrapper (`llm.py`)

In [5]:
%%writefile llm.py
"""
llm.py
------
Thin wrapper around the Gemini API. Everything that talks to the model
goes through GeminiLLM so the rest of the codebase doesn't need to know
or care which SDK/model is behind it.

Deliberately NOT doing any scoring/ranking/arithmetic here - that's all
in scoring.py, in plain Python, so it's deterministic and reproducible.
"""

import json
import re
import time

from google import genai

import config


class GeminiError(Exception):
    """Raised when Gemini can't be reached or returns something unusable."""
    pass


class GeminiLLM:
    def __init__(self, api_key: str, model: str = None):
        if not api_key:
            raise GeminiError(
                "No Gemini API key was provided. In Colab, set a secret named "
                "GEMINI_API_KEY (Tools -> Secrets) and grant this notebook access."
            )
        self.model = model or config.GEMINI_MODEL
        self._client = genai.Client(api_key=api_key)
        self.call_count = 0  # cheap way to track free-tier usage during a run

    def generate(self, prompt: str) -> str:
        """Send a prompt to Gemini and return the raw text response.

        Retries a handful of times with backoff since free-tier keys get
        rate limited fairly easily. If everything fails, raises GeminiError
        rather than letting the whole agent run crash silently.
        """
        last_error = None
        for attempt in range(1, config.GEMINI_MAX_RETRIES + 1):
            try:
                self.call_count += 1
                response = self._client.models.generate_content(
                    model=self.model,
                    contents=prompt,
                )
                text = getattr(response, "text", None)
                if not text:
                    raise GeminiError("Gemini returned an empty response.")
                return text
            except Exception as exc:  # noqa: BLE001 - we want to catch/retry broadly here
                last_error = exc
                if attempt < config.GEMINI_MAX_RETRIES:
                    delay = config.GEMINI_RETRY_BASE_DELAY_SECONDS * attempt
                    print(f"  ⚠ Gemini call failed (attempt {attempt}), retrying in {delay}s... ({exc})")
                    time.sleep(delay)
        raise GeminiError(f"Gemini call failed after {config.GEMINI_MAX_RETRIES} attempts: {last_error}")

    def generate_json(self, prompt: str, retries: int = 1) -> dict:
        """Ask Gemini for JSON and parse it, handling the usual markdown-fence mess.

        If parsing fails, we retry once with a stricter follow-up prompt
        before giving up gracefully.
        """
        raw = self.generate(prompt)
        parsed = _extract_json(raw)
        if parsed is not None:
            return parsed

        if retries > 0:
            stricter_prompt = (
                prompt
                + "\n\nIMPORTANT: Your last response could not be parsed as JSON. "
                "Reply with ONLY valid JSON. No markdown fences, no commentary, no preamble."
            )
            raw2 = self.generate(stricter_prompt)
            parsed2 = _extract_json(raw2)
            if parsed2 is not None:
                return parsed2

        raise GeminiError(
            "Could not parse JSON from Gemini's response after retrying. "
            f"Raw response started with: {raw[:200]!r}"
        )


def _extract_json(text: str):
    """Try hard to pull a JSON object out of an LLM response.

    Handles: plain JSON, ```json fenced blocks, ``` fenced blocks with no
    language tag, and JSON with trailing chatter before/after it.
    """
    text = text.strip()

    # Case 1: fenced code block
    fence_match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    candidates = []
    if fence_match:
        candidates.append(fence_match.group(1).strip())

    # Case 2: whole string as-is
    candidates.append(text)

    # Case 3: first {...} or [...] block found anywhere in the text
    brace_match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if brace_match:
        candidates.append(brace_match.group(1).strip())

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, TypeError):
            continue

    return None


Overwriting llm.py


## 6. Resume parser (`parser.py`)

In [6]:
%%writefile parser.py
"""
parser.py
---------
Everything to do with getting raw text off disk and into the agent.
No Gemini calls in here on purpose - this is pure file I/O so it can be
unit tested without an API key.
"""

import os

import config


class ResumeLoadError(Exception):
    pass


def load_resume(path: str) -> dict:
    """Load a single resume file and return a plain dict with its text.

    Supports .txt, .md, .pdf. Anything else raises ResumeLoadError so the
    caller can decide to skip it instead of crashing the whole batch.
    """
    if not os.path.exists(path):
        raise ResumeLoadError(f"File not found: {path}")

    filename = os.path.basename(path)
    candidate_id = os.path.splitext(filename)[0]
    ext = os.path.splitext(filename)[1].lower()

    if ext in (".txt", ".md"):
        text = _load_text_file(path)
    elif ext == ".pdf":
        text = _load_pdf_file(path)
    else:
        raise ResumeLoadError(f"Unsupported file type '{ext}' for {filename}")

    text = text.strip()
    if not text:
        raise ResumeLoadError(f"{filename} appears to be empty.")

    if len(text) > config.MAX_RESUME_CHARS:
        text = text[: config.MAX_RESUME_CHARS]

    return {
        "candidate_id": candidate_id,
        "filename": filename,
        "text": text,
    }


def load_resumes_from_dir(directory: str) -> tuple[list, list]:
    """Load every supported resume in a directory.

    Returns (loaded, errors) so the caller can report skipped files
    without the whole run failing because of one bad PDF.
    """
    loaded = []
    errors = []

    if not os.path.isdir(directory):
        raise ResumeLoadError(f"Resume directory not found: {directory}")

    filenames = sorted(os.listdir(directory))
    for filename in filenames:
        full_path = os.path.join(directory, filename)
        if not os.path.isfile(full_path):
            continue
        try:
            resume = load_resume(full_path)
            loaded.append(resume)
        except ResumeLoadError as exc:
            errors.append({"filename": filename, "error": str(exc)})

        if len(loaded) >= config.MAX_CANDIDATES:
            break

    return loaded, errors


def _load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def _load_pdf_file(path: str) -> str:
    try:
        from pypdf import PdfReader
    except ImportError as exc:
        raise ResumeLoadError(
            "pypdf is not installed. Run: pip install pypdf"
        ) from exc

    try:
        reader = PdfReader(path)
        pages_text = []
        for page in reader.pages:
            page_text = page.extract_text() or ""
            pages_text.append(page_text)
        text = "\n".join(pages_text).strip()
        if not text:
            raise ResumeLoadError(
                f"⚠ Could not extract text from candidate PDF: {os.path.basename(path)} "
                "(it may be a scanned image without a text layer)."
            )
        return text
    except ResumeLoadError:
        raise
    except Exception as exc:  # noqa: BLE001
        raise ResumeLoadError(
            f"⚠ Could not extract text from candidate PDF: {os.path.basename(path)} ({exc})"
        ) from exc


Overwriting parser.py


## 7-11. Job analyzer, candidate profile extractor, requirement matcher, gap analysis, evidence tracking (`tools.py`)

These are the pieces that actually call Gemini. Everything here is batched:
one call analyzes the job, one call extracts *all* candidate profiles, one
call matches *all* candidates against every requirement. That is the whole
free-tier budget for extraction - only the final report summary uses a 4th call.


In [7]:
%%writefile tools.py
"""
tools.py
--------
The "tools" the agent calls. Each function here does one job:
- ask Gemini to extract structure from unstructured text, or
- do local bookkeeping (privacy filtering) that must never touch an LLM.

Four Gemini calls total in a normal run, all batched:
  1. analyze_job_description        -> one call
  2. extract_candidate_profiles     -> one call for ALL resumes together
  3. evaluate_candidates            -> one call for ALL candidates together
  4. synthesize_report              -> one call

That's the free-tier budget this whole project is built around.
"""

import json

import config
from llm import GeminiLLM


# ---------------------------------------------------------------------
# 1. Job requirement extraction
# ---------------------------------------------------------------------

def analyze_job_description(llm: GeminiLLM, job_description: str) -> dict:
    prompt = f"""You are analyzing a job description for a recruiting tool.
Extract structured requirements. Only use information present in the text
below - do not invent requirements that aren't there.

Return ONLY valid JSON in exactly this shape:
{{
  "role": "<job title>",
  "mandatory_skills": ["skill", ...],
  "preferred_skills": ["skill", ...],
  "minimum_experience_years": <number or null>,
  "responsibilities": ["responsibility", ...]
}}

Job description:
\"\"\"
{job_description}
\"\"\"
"""
    result = llm.generate_json(prompt)
    result.setdefault("role", "unknown")
    result.setdefault("mandatory_skills", [])
    result.setdefault("preferred_skills", [])
    result.setdefault("minimum_experience_years", None)
    result.setdefault("responsibilities", [])
    return result


# ---------------------------------------------------------------------
# 2. Candidate profile extraction (batched across all resumes)
# ---------------------------------------------------------------------

def extract_candidate_profiles(llm: GeminiLLM, resumes: list) -> list:
    """resumes: list of {'candidate_id', 'filename', 'text'}.

    Sends all resumes in ONE prompt and asks for one profile per resume,
    to keep Gemini usage inside the free tier budget.
    """
    resume_blocks = []
    for r in resumes:
        resume_blocks.append(f'--- RESUME ID: {r["candidate_id"]} ---\n{r["text"]}')
    joined = "\n\n".join(resume_blocks)

    prompt = f"""You are extracting structured candidate profiles from resumes for a
recruiting tool. Only extract information EXPLICITLY present in each
resume's text. If something is not mentioned, use "unknown" (for single
values) or an empty list (for lists). Do NOT guess or infer.

Do NOT extract or mention: age, gender, race, ethnicity, religion,
disability, health information, marital status, sexual orientation,
nationality, or anything about a photograph. If such information
appears in the text, ignore it entirely - it has no place in this profile.

Return ONLY a valid JSON array. One object per resume, in this exact shape:
[
  {{
    "candidate_id": "<the RESUME ID given above>",
    "skills": ["skill", ...],
    "experience_years": <number or "unknown">,
    "projects": ["short project description", ...],
    "education": ["degree/field", ...],
    "certifications": ["cert", ...],
    "evidence_notes": ["short quote or paraphrase supporting a skill claim", ...]
  }},
  ...
]

Resumes:
{joined}
"""
    result = llm.generate_json(prompt)
    if isinstance(result, dict) and "candidates" in result:
        result = result["candidates"]
    if not isinstance(result, list):
        raise ValueError("Expected a JSON array of candidate profiles from Gemini.")

    profiles = []
    for item in result:
        profiles.append(_strip_prohibited_fields(item))
    return profiles


def _strip_prohibited_fields(profile: dict) -> dict:
    """Defense in depth: even though the prompt tells Gemini not to
    include sensitive fields, we strip them locally too, so a scoring
    bug or a model hiccup can never let a prohibited field leak into
    scoring or the report.
    """
    cleaned = {}
    for key, value in profile.items():
        if key.lower() in config.PROHIBITED_FIELDS:
            continue
        cleaned[key] = value
    cleaned.setdefault("skills", [])
    cleaned.setdefault("experience_years", "unknown")
    cleaned.setdefault("projects", [])
    cleaned.setdefault("education", [])
    cleaned.setdefault("certifications", [])
    cleaned.setdefault("evidence_notes", [])
    return cleaned


# ---------------------------------------------------------------------
# 3. Requirement matching + gap analysis (batched across all candidates)
# ---------------------------------------------------------------------

def evaluate_candidates(llm: GeminiLLM, job_requirements: dict, profiles: list) -> dict:
    """One call that matches every candidate against every requirement
    and returns match status + evidence per requirement per candidate.

    This is the semantic-matching step (e.g. recognizing "built REST
    services with Flask" as partial evidence for "FastAPI"). The score
    itself is NOT computed here - scoring.py does that deterministically
    from the MATCH/PARTIAL_MATCH/NO_EVIDENCE labels this returns.
    """
    mandatory = job_requirements.get("mandatory_skills", [])
    preferred = job_requirements.get("preferred_skills", [])

    profiles_json = json.dumps(profiles, indent=2)
    prompt = f"""You are matching candidates against job requirements for a recruiting
tool. For EVERY requirement below, and EVERY candidate, decide a status:

- "MATCH": the candidate's profile clearly shows this skill/requirement.
- "PARTIAL_MATCH": related or adjacent experience, but not a clean match.
- "NO_EVIDENCE": the profile does not mention it. This does NOT mean the
  candidate lacks the skill - only that there is no evidence in the resume.

Never write "does not have" - always use NO_EVIDENCE for absence of
mention. For every MATCH or PARTIAL_MATCH, include a short evidence
string paraphrased from the candidate's profile (skills/projects/notes).
For NO_EVIDENCE, evidence should be an empty string "".

Mandatory requirements: {json.dumps(mandatory)}
Preferred requirements: {json.dumps(preferred)}

Candidate profiles:
{profiles_json}

Return ONLY valid JSON in exactly this shape:
{{
  "candidate_01": {{
    "mandatory_matches": [
      {{"requirement": "Python", "status": "MATCH", "evidence": "..."}},
      ...
    ],
    "preferred_matches": [
      {{"requirement": "Docker", "status": "NO_EVIDENCE", "evidence": ""}},
      ...
    ],
    "strengths": ["short phrase", ...],
    "gaps": ["Not evidenced in the provided resume: <requirement>", ...]
  }},
  ...
}}
Include one key per candidate_id given above.
"""
    result = llm.generate_json(prompt)
    if not isinstance(result, dict):
        raise ValueError("Expected a JSON object keyed by candidate_id from Gemini.")
    return result


# ---------------------------------------------------------------------
# 4. Final report synthesis
# ---------------------------------------------------------------------

def synthesize_report_summary(llm: GeminiLLM, job_requirements: dict, ranked_summaries: list) -> str:
    """One call to write a short, plain-language executive summary.
    Everything factual (scores, ranks, coverage) is computed in Python
    and just handed to Gemini as context - Gemini is only writing prose
    here, not deciding any numbers.
    """
    prompt = f"""Write a short (150-250 word) executive summary for a recruiting
decision-support report. Be factual and neutral. Do NOT declare any
candidate "the best hire" or say a candidate "will succeed" - use
language like "recommended for human review" and "strong match based on
provided evidence". Do not mention age, gender, or any protected
characteristic. End by reminding the reader that final hiring decisions
require human review.

Role: {job_requirements.get('role', 'unknown')}

Ranked candidate summaries (already scored deterministically in Python):
{json.dumps(ranked_summaries, indent=2)}

Write only the summary text, no headers, no markdown title.
"""
    return llm.generate(prompt).strip()


# ---------------------------------------------------------------------
# Local validation (no LLM involved)
# ---------------------------------------------------------------------

def validate_profile_privacy(profile: dict) -> list:
    """Returns a list of violation strings if any prohibited field is
    present. Should always be empty after _strip_prohibited_fields runs,
    but kept as a standalone checkable function for tests / the agent's
    quality-check step.
    """
    violations = []
    for key in profile.keys():
        if key.lower() in config.PROHIBITED_FIELDS:
            violations.append(key)
    return violations


Overwriting tools.py


## 12. Deterministic scoring engine (`scoring.py`)

In [8]:
%%writefile scoring.py
"""
scoring.py
----------
Deterministic, reproducible scoring. No LLM calls anywhere in this file
- that's the whole point. Given the same matches twice, you get the same
score twice. Gemini is good at reading resumes; it's not the thing that
should decide 78 vs 81.
"""

import config


def _validate_weights():
    total = sum(config.SCORING_WEIGHTS.values())
    if abs(total - 1.0) > 0.001:
        raise ValueError(f"SCORING_WEIGHTS must sum to 1.0, got {total}")


_validate_weights()


def calculate_skill_score(matches: list, weight: float) -> float:
    """matches: list of {'status': 'MATCH'|'PARTIAL_MATCH'|'NO_EVIDENCE'}.

    MATCH counts full, PARTIAL_MATCH counts half, NO_EVIDENCE counts zero.
    Returns points out of (weight * 100), capped so rounding can never
    push it over the max.
    """
    if not matches:
        return 0.0

    total_points = 0.0
    for m in matches:
        status = m.get("status", "NO_EVIDENCE")
        if status == "MATCH":
            total_points += 1.0
        elif status == "PARTIAL_MATCH":
            total_points += 0.5
        # NO_EVIDENCE contributes 0

    ratio = total_points / len(matches)
    max_points = weight * 100
    return round(min(ratio * max_points, max_points), 2)


def calculate_experience_score(candidate_years, required_years, weight: float) -> float:
    """Full credit if candidate meets/exceeds requirement, partial credit
    scaled linearly if below it, zero if years unknown.
    """
    max_points = weight * 100
    if candidate_years in (None, "unknown"):
        return 0.0
    if required_years in (None, "unknown") or required_years == 0:
        return max_points  # no explicit requirement -> don't penalize

    try:
        candidate_years = float(candidate_years)
        required_years = float(required_years)
    except (TypeError, ValueError):
        return 0.0

    if candidate_years >= required_years:
        return round(max_points, 2)

    ratio = max(candidate_years / required_years, 0.0)
    return round(min(ratio * max_points, max_points), 2)


def calculate_project_score(projects: list, relevant_keywords: list, weight: float) -> float:
    """Rough but transparent: how many listed projects mention at least
    one relevant keyword, out of total projects considered (capped at 3
    so a candidate can't just pad a giant project list).
    """
    max_points = weight * 100
    if not projects:
        return 0.0

    keywords_lower = [k.lower() for k in relevant_keywords]
    considered = projects[:3]
    relevant_count = 0
    for project in considered:
        project_text = str(project).lower()
        if any(kw in project_text for kw in keywords_lower):
            relevant_count += 1

    ratio = relevant_count / len(considered)
    return round(min(ratio * max_points, max_points), 2)


def calculate_evidence_score(matches: list, weight: float) -> float:
    """Rewards matches that come with actual evidence text attached,
    not just a bare MATCH label.
    """
    max_points = weight * 100
    matched = [m for m in matches if m.get("status") in ("MATCH", "PARTIAL_MATCH")]
    if not matched:
        return 0.0

    with_evidence = sum(1 for m in matched if m.get("evidence", "").strip())
    ratio = with_evidence / len(matched)
    return round(min(ratio * max_points, max_points), 2)


def calculate_total_score(components: dict) -> dict:
    """components: dict with keys mandatory_skills, preferred_skills,
    experience, projects, evidence - each already a point value (not a
    ratio) that respects its own weight's max.

    Returns a breakdown dict plus the total, clamped to [0, 100].
    """
    weights = config.SCORING_WEIGHTS
    breakdown = {}
    total = 0.0
    for key, weight in weights.items():
        max_points = round(weight * 100, 2)
        value = components.get(key, 0.0)
        value = max(0.0, min(value, max_points))  # never exceed max for that component
        breakdown[key] = {"score": value, "max": max_points}
        total += value

    total = round(max(0.0, min(total, 100.0)), 2)
    breakdown["total"] = total
    return breakdown


def mandatory_coverage_ratio(matches: list) -> float:
    """Fraction of mandatory requirements that are at least MATCH or
    PARTIAL_MATCH. Used for the shortlist threshold, separate from score.
    """
    if not matches:
        return 0.0
    covered = sum(1 for m in matches if m.get("status") in ("MATCH", "PARTIAL_MATCH"))
    return covered / len(matches)


def rank_candidates(evaluations: list) -> list:
    """Sort candidates by total score, descending. Ties broken by
    mandatory coverage, then candidate_id for stability.
    """
    def sort_key(ev):
        return (
            -ev["score_breakdown"]["total"],
            -ev.get("mandatory_coverage", 0.0),
            ev["candidate_id"],
        )

    ranked = sorted(evaluations, key=sort_key)
    for i, ev in enumerate(ranked, start=1):
        ev["rank"] = i
    return ranked


def create_shortlist(ranked_evaluations: list, size: int = None, min_coverage: float = None) -> list:
    """Take the top N candidates that also clear the mandatory-coverage bar.

    Score alone isn't enough to shortlist someone - they also need to
    genuinely cover the mandatory requirements, otherwise a candidate
    with great "nice to have" skills but no mandatory-skill evidence
    could sneak in on preferred/experience/project points.
    """
    size = config.SHORTLIST_SIZE if size is None else size
    min_coverage = config.MIN_MANDATORY_COVERAGE if min_coverage is None else min_coverage

    eligible = [
        ev for ev in ranked_evaluations
        if ev.get("mandatory_coverage", 0.0) >= min_coverage
    ]
    return eligible[:size]


Overwriting scoring.py


## 13-16. Agent state, loop, evidence validator, report generator (`recruiting_agent.py`)

This is the actual agent: an explicit `RecruitingState` dataclass, a plan
built from the job requirements, and a bounded loop (`MAX_ITERATIONS = 4`)
over a fixed action list, ending with a self-check before it ranks anyone.


In [9]:
%%writefile recruiting_agent.py
"""
recruiting_agent.py
--------------------
The agent itself: state, a dynamically-generated plan, a bounded loop
over a fixed set of actions, and a self-check before it's willing to
call the run finished.

This is deliberately NOT "resume -> Gemini -> score". Gemini is used for
three narrow, batched extraction/writing jobs; everything about how a
score is computed, who gets ranked where, and who makes the shortlist
is plain deterministic Python in scoring.py.
"""

import json
import os
from dataclasses import dataclass, field
from datetime import datetime, timezone

import config
import parser
import scoring
import tools
from llm import GeminiLLM, GeminiError


# ---------------------------------------------------------------------
# State
# ---------------------------------------------------------------------

@dataclass
class RecruitingState:
    job_description: str
    job_requirements: dict = field(default_factory=dict)
    plan: list = field(default_factory=list)
    resumes: list = field(default_factory=list)              # raw loaded text
    load_errors: list = field(default_factory=list)          # files that failed to load
    candidate_profiles: list = field(default_factory=list)   # structured, privacy-filtered
    match_results: dict = field(default_factory=dict)        # candidate_id -> matches
    evaluations: list = field(default_factory=list)          # scored + ranked
    shortlisted: list = field(default_factory=list)
    quality_issues: list = field(default_factory=list)
    trace: list = field(default_factory=list)
    gemini_calls: int = 0
    iterations: int = 0
    final_report_md: str = ""
    final_report_json: dict = field(default_factory=dict)


ACTIONS = [
    "ANALYZE_JOB",
    "CREATE_PLAN",
    "PARSE_RESUMES",
    "EXTRACT_PROFILES",
    "MATCH_REQUIREMENTS",
    "SCORE_AND_RANK",
    "VALIDATE_EVIDENCE",
    "GENERATE_REPORT",
    "FINISH",
]


class RecruitingAgent:
    def __init__(self, llm: GeminiLLM):
        self.llm = llm

    # -------------------------------------------------------------
    # Public entry point
    # -------------------------------------------------------------
    def run(self, job_description: str, resume_dir: str) -> RecruitingState:
        state = RecruitingState(job_description=job_description)
        action_index = 0

        while state.iterations < config.MAX_ITERATIONS and action_index < len(ACTIONS):
            action = ACTIONS[action_index]
            self._log(state, f"iteration {state.iterations + 1} -> action: {action}")

            if action == "ANALYZE_JOB":
                self._analyze_job(state)
            elif action == "CREATE_PLAN":
                self._create_plan(state)
            elif action == "PARSE_RESUMES":
                self._parse_resumes(state, resume_dir)
            elif action == "EXTRACT_PROFILES":
                self._extract_profiles(state)
            elif action == "MATCH_REQUIREMENTS":
                self._match_requirements(state)
            elif action == "SCORE_AND_RANK":
                self._score_and_rank(state)
            elif action == "VALIDATE_EVIDENCE":
                self._validate_evidence(state)
            elif action == "GENERATE_REPORT":
                self._generate_report(state)
            elif action == "FINISH":
                self._log(state, "run complete")
                break

            action_index += 1
            state.iterations += 1

        state.gemini_calls = self.llm.call_count
        return state

    # -------------------------------------------------------------
    # Actions
    # -------------------------------------------------------------
    def _analyze_job(self, state: RecruitingState):
        print("\n🧠 Step 1 — Analyzing job requirements")
        state.job_requirements = tools.analyze_job_description(self.llm, state.job_description)
        n_mandatory = len(state.job_requirements.get("mandatory_skills", []))
        n_preferred = len(state.job_requirements.get("preferred_skills", []))
        print(f"✓ {n_mandatory} mandatory requirements")
        print(f"✓ {n_preferred} preferred requirements")
        self._log(state, f"extracted {n_mandatory} mandatory / {n_preferred} preferred requirements")

    def _create_plan(self, state: RecruitingState):
        print("\n📋 Step 2 — Creating evaluation plan")
        role = state.job_requirements.get("role", "the role")
        state.plan = [
            f"Analyze requirements for {role}.",
            "Extract mandatory and preferred criteria.",
            "Parse all candidate resumes.",
            "Build structured candidate profiles (skills, experience, projects only).",
            "Match each profile against every requirement with evidence.",
            "Calculate deterministic, weighted scores in Python.",
            "Run a quality/evidence check before ranking.",
            "Rank candidates and build a shortlist for human review.",
            "Produce an explainable report.",
        ]
        print("✓ Evaluation plan created")
        self._log(state, "plan created with 9 steps")

    def _parse_resumes(self, state: RecruitingState, resume_dir: str):
        print("\n📄 Step 3 — Parsing resumes")
        loaded, errors = parser.load_resumes_from_dir(resume_dir)
        state.resumes = loaded
        state.load_errors = errors
        print(f"✓ {len(loaded)} resumes loaded")
        if errors:
            for e in errors:
                print(f"  ⚠ Skipped {e['filename']}: {e['error']}")
        self._log(state, f"loaded {len(loaded)} resumes, {len(errors)} errors")

    def _extract_profiles(self, state: RecruitingState):
        print("\n🧩 Step 4 — Extracting candidate profiles")
        if not state.resumes:
            print("  ⚠ No resumes to extract profiles from.")
            return
        state.candidate_profiles = tools.extract_candidate_profiles(self.llm, state.resumes)
        print(f"✓ {len(state.candidate_profiles)} candidate profiles extracted")
        self._log(state, f"extracted {len(state.candidate_profiles)} profiles")

    def _match_requirements(self, state: RecruitingState):
        print("\n🔍 Step 5 — Matching candidates against requirements")
        if not state.candidate_profiles:
            print("  ⚠ No profiles to match.")
            return
        state.match_results = tools.evaluate_candidates(
            self.llm, state.job_requirements, state.candidate_profiles
        )
        print(f"✓ Candidate evaluations created for {len(state.match_results)} candidates")
        self._log(state, f"matched {len(state.match_results)} candidates against requirements")

    def _score_and_rank(self, state: RecruitingState):
        print("\n📊 Step 6 — Calculating deterministic scores")
        weights = config.SCORING_WEIGHTS
        job_keywords = (
            state.job_requirements.get("mandatory_skills", [])
            + state.job_requirements.get("preferred_skills", [])
        )
        min_experience = state.job_requirements.get("minimum_experience_years")

        evaluations = []
        profiles_by_id = {p["candidate_id"]: p for p in state.candidate_profiles}

        for candidate_id, match in state.match_results.items():
            profile = profiles_by_id.get(candidate_id, {})
            mandatory_matches = match.get("mandatory_matches", [])
            preferred_matches = match.get("preferred_matches", [])
            all_matches = mandatory_matches + preferred_matches

            mandatory_score = scoring.calculate_skill_score(mandatory_matches, weights["mandatory_skills"])
            preferred_score = scoring.calculate_skill_score(preferred_matches, weights["preferred_skills"])
            experience_score = scoring.calculate_experience_score(
                profile.get("experience_years"), min_experience, weights["experience"]
            )
            project_score = scoring.calculate_project_score(
                profile.get("projects", []), job_keywords, weights["projects"]
            )
            evidence_score = scoring.calculate_evidence_score(all_matches, weights["evidence"])

            breakdown = scoring.calculate_total_score({
                "mandatory_skills": mandatory_score,
                "preferred_skills": preferred_score,
                "experience": experience_score,
                "projects": project_score,
                "evidence": evidence_score,
            })

            evaluations.append({
                "candidate_id": candidate_id,
                "profile": profile,
                "mandatory_matches": mandatory_matches,
                "preferred_matches": preferred_matches,
                "strengths": match.get("strengths", []),
                "gaps": match.get("gaps", []),
                "mandatory_coverage": round(scoring.mandatory_coverage_ratio(mandatory_matches), 2),
                "score_breakdown": breakdown,
            })

        evaluations = scoring.rank_candidates(evaluations)
        state.evaluations = evaluations
        state.shortlisted = scoring.create_shortlist(evaluations)

        print("✓ Scores calculated")
        self._log(state, f"scored and ranked {len(evaluations)} candidates")

    def _validate_evidence(self, state: RecruitingState):
        print("\n🔎 Step 7 — Validating evidence")
        issues = []

        mandatory_reqs = set(state.job_requirements.get("mandatory_skills", []))

        for ev in state.evaluations:
            cid = ev["candidate_id"]

            evaluated = {m["requirement"] for m in ev["mandatory_matches"]}
            missing = mandatory_reqs - evaluated
            if missing:
                issues.append(f"{cid}: mandatory requirements not evaluated: {sorted(missing)}")

            for m in ev["mandatory_matches"] + ev["preferred_matches"]:
                if m["status"] in ("MATCH", "PARTIAL_MATCH") and not m.get("evidence", "").strip():
                    issues.append(f"{cid}: '{m['requirement']}' marked {m['status']} with no evidence text")

            total = ev["score_breakdown"]["total"]
            if not (0 <= total <= 100):
                issues.append(f"{cid}: score {total} out of bounds")

            violations = tools.validate_profile_privacy(ev["profile"])
            if violations:
                issues.append(f"{cid}: prohibited fields present in profile: {violations}")

        state.quality_issues = issues
        if issues:
            print(f"  ⚠ {len(issues)} issue(s) found:")
            for i in issues:
                print(f"    - {i}")
        else:
            print("✓ Evidence validation completed — no issues found")
        self._log(state, f"quality check found {len(issues)} issue(s)")

    def _generate_report(self, state: RecruitingState):
        print("\n🏆 Step 8 — Ranking candidates and generating report")

        ranked_summaries = [
            {
                "candidate_id": ev["candidate_id"],
                "rank": ev["rank"],
                "score": ev["score_breakdown"]["total"],
                "mandatory_coverage": ev["mandatory_coverage"],
                "strengths": ev["strengths"],
                "gaps": ev["gaps"],
            }
            for ev in state.evaluations
        ]

        summary_text = ""
        try:
            summary_text = tools.synthesize_report_summary(
                self.llm, state.job_requirements, ranked_summaries
            )
        except GeminiError as exc:
            summary_text = (
                "Executive summary could not be generated automatically "
                f"({exc}). See the candidate ranking and detailed evaluations below."
            )

        state.final_report_md = _build_markdown_report(state, summary_text)
        state.final_report_json = _build_json_report(state)

        print("✓ Report generated")
        self._log(state, "final report generated")

    # -------------------------------------------------------------
    def _log(self, state: RecruitingState, message: str):
        stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
        state.trace.append(f"[{stamp}] {message}")


# ---------------------------------------------------------------------
# Report builders (pure Python, no Gemini)
# ---------------------------------------------------------------------

def _status_symbol(status: str) -> str:
    return {"MATCH": "✓", "PARTIAL_MATCH": "~", "NO_EVIDENCE": "-"}.get(status, "?")


def _build_markdown_report(state: RecruitingState, summary_text: str) -> str:
    lines = []
    role = state.job_requirements.get("role", "unknown")
    lines.append(f"# Recruiting Report — {role}")
    lines.append("")
    lines.append(f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(summary_text or "_No summary available._")
    lines.append("")

    lines.append("## Job Requirements")
    lines.append(f"**Role:** {role}")
    lines.append("")
    lines.append("**Mandatory:** " + ", ".join(state.job_requirements.get("mandatory_skills", [])) or "_none extracted_")
    lines.append("")
    lines.append("**Preferred:** " + ", ".join(state.job_requirements.get("preferred_skills", [])) or "_none extracted_")
    lines.append("")
    min_exp = state.job_requirements.get("minimum_experience_years")
    lines.append(f"**Minimum experience:** {min_exp if min_exp is not None else 'not specified'} years")
    lines.append("")

    lines.append("## Candidate Ranking")
    lines.append("")
    for ev in state.evaluations:
        lines.append(f"{ev['rank']}. Candidate `{ev['candidate_id']}` — {ev['score_breakdown']['total']}/100")
    lines.append("")

    lines.append("## Recommended Shortlist")
    lines.append("")
    if state.shortlisted:
        for ev in state.shortlisted:
            lines.append(f"- `{ev['candidate_id']}` — {ev['score_breakdown']['total']}/100 — recommended for human review")
    else:
        lines.append("_No candidates cleared the minimum mandatory-coverage threshold._")
    lines.append("")

    lines.append("## Detailed Candidate Evaluation")
    for ev in state.evaluations:
        lines.append("")
        lines.append(f"### Candidate `{ev['candidate_id']}`")
        lines.append(f"**Overall Score:** {ev['score_breakdown']['total']}/100")
        lines.append("**Status:** Recommended for Human Review")
        lines.append(f"**Mandatory Skill Coverage:** {round(ev['mandatory_coverage'] * 100)}%")
        lines.append("")
        lines.append("**Score Breakdown:**")
        for key, val in ev["score_breakdown"].items():
            if key == "total":
                continue
            lines.append(f"- {key.replace('_', ' ').title()}: {val['score']}/{val['max']}")
        lines.append("")
        lines.append("**Mandatory Requirements:**")
        for m in ev["mandatory_matches"]:
            symbol = _status_symbol(m["status"])
            evidence = f" — {m['evidence']}" if m.get("evidence") else ""
            lines.append(f"- {symbol} {m['requirement']} ({m['status']}){evidence}")
        lines.append("")
        lines.append("**Preferred Requirements:**")
        for m in ev["preferred_matches"]:
            symbol = _status_symbol(m["status"])
            evidence = f" — {m['evidence']}" if m.get("evidence") else ""
            lines.append(f"- {symbol} {m['requirement']} ({m['status']}){evidence}")
        lines.append("")
        lines.append("**Strengths:**")
        for s in ev["strengths"]:
            lines.append(f"- {s}")
        lines.append("")
        lines.append("**Potential Gaps:**")
        for g in ev["gaps"]:
            lines.append(f"- {g}")
        lines.append("")
        lines.append("**Recommendation:** Strong match based on provided evidence. Recommended for human review.")

    lines.append("")
    lines.append("## Human Review Considerations")
    lines.append(
        "- Missing evidence for a requirement does not mean the candidate lacks that skill — "
        "it means the resume did not mention it.\n"
        "- Scores reflect resume content only, not interview performance or references.\n"
        "- Recruiters should verify high-impact claims directly with candidates."
    )
    lines.append("")
    lines.append("## Limitations")
    lines.append(
        "- Resume quality directly affects extraction quality.\n"
        "- LLM-based extraction can contain errors and should be spot-checked.\n"
        "- Scores depend on the configurable weighting in `config.py`.\n"
        "- This prototype is not a production hiring system."
    )
    lines.append("")
    lines.append("---")
    lines.append(config.DISCLAIMER)

    return "\n".join(lines)


def _build_json_report(state: RecruitingState) -> dict:
    return {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "job": state.job_requirements,
        "gemini_calls": state.gemini_calls,
        "candidates_processed": len(state.candidate_profiles),
        "load_errors": state.load_errors,
        "quality_issues": state.quality_issues,
        "candidates": [
            {
                "candidate_id": ev["candidate_id"],
                "rank": ev["rank"],
                "score": ev["score_breakdown"]["total"],
                "score_breakdown": ev["score_breakdown"],
                "mandatory_coverage": ev["mandatory_coverage"],
                "mandatory_matches": ev["mandatory_matches"],
                "preferred_matches": ev["preferred_matches"],
                "strengths": ev["strengths"],
                "gaps": ev["gaps"],
                "priority": "SHORTLISTED" if ev in state.shortlisted else "REVIEW",
            }
            for ev in state.evaluations
        ],
        "shortlist": [ev["candidate_id"] for ev in state.shortlisted],
        "disclaimer": config.DISCLAIMER,
    }


# ---------------------------------------------------------------------
# Convenience runner used by both the CLI and the notebook
# ---------------------------------------------------------------------

def run_demo(api_key: str, job_description_path: str = None, resume_dir: str = None):
    job_description_path = job_description_path or config.JOB_DESCRIPTION_PATH
    resume_dir = resume_dir or config.RESUME_DIR

    with open(job_description_path, "r", encoding="utf-8") as f:
        job_description = f.read()

    llm = GeminiLLM(api_key=api_key)
    agent = RecruitingAgent(llm)

    print("=" * 56)
    print("🤖 AGENTIC RECRUITING AGENT")
    print("=" * 56)
    print(f"\n💼 Job description loaded from: {job_description_path}")

    state = agent.run(job_description, resume_dir)

    print("\n" + "=" * 56)
    print("SHORTLIST")
    print("=" * 56)
    for ev in state.shortlisted:
        print(f"{ev['rank']}. {ev['candidate_id']} — {ev['score_breakdown']['total']}/100")
    print(f"\nGemini calls: {state.gemini_calls}")
    print(f"Candidates processed: {len(state.candidate_profiles)}")
    print(f"Iterations: {state.iterations}")
    print(f"\n⚠ {config.DISCLAIMER}")

    return state


def export_outputs(state: RecruitingState, output_dir: str = None):
    output_dir = output_dir or config.OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)

    json_path = os.path.join(output_dir, "candidate_rankings.json")
    md_path = os.path.join(output_dir, "recruiting_report.md")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(state.final_report_json, f, indent=2)

    with open(md_path, "w", encoding="utf-8") as f:
        f.write(state.final_report_md)

    return json_path, md_path


Overwriting recruiting_agent.py


## 17. Create the synthetic job description

In [10]:
import os
os.makedirs("data/resumes", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

job_description_text = """AI Engineer

We are looking for an AI Engineer to build production-oriented AI applications.

Required:
- Python
- FastAPI
- REST APIs
- LLM application development
- RAG
- Git
- 2+ years software engineering experience

Preferred:
- Docker
- AWS
- Vector databases
- Machine learning experience

Responsibilities:
- Build AI-powered applications
- Develop backend APIs
- Integrate LLMs
- Build retrieval-augmented generation systems
- Write maintainable Python code
- Collaborate with engineering teams
"""

with open("data/job_description.txt", "w") as f:
    f.write(job_description_text)

print("✓ data/job_description.txt written")
print(job_description_text)


✓ data/job_description.txt written
AI Engineer

We are looking for an AI Engineer to build production-oriented AI applications.

Required:
- Python
- FastAPI
- REST APIs
- LLM application development
- RAG
- Git
- 2+ years software engineering experience

Preferred:
- Docker
- AWS
- Vector databases
- Machine learning experience

Responsibilities:
- Build AI-powered applications
- Develop backend APIs
- Integrate LLMs
- Build retrieval-augmented generation systems
- Write maintainable Python code
- Collaborate with engineering teams



## 18. Create the five synthetic candidate resumes

All fictional — no real people, no real resumes.

In [11]:
resumes = {
    "candidate_01.txt": open("data/resumes/candidate_01.txt", "w") if False else None,
}

candidate_01 = """Priya Nair
Backend Software Engineer

Summary
Backend engineer with 3 years of experience building web services in Python.
Comfortable owning a service from design to deployment.

Skills
Python, FastAPI, Flask, PostgreSQL, REST APIs, Git, Docker, pytest

Experience
Backend Engineer, Fintrack Labs (2022 - Present)
- Built and maintained REST APIs using FastAPI for a transaction-tracking product
- Migrated a legacy Flask service to FastAPI, cutting response times by 30%
- Wrote unit and integration tests with pytest, kept coverage above 85%
- Used Git for version control across a team of 6 engineers

Software Engineer, Loopstack (2021 - 2022)
- Developed internal REST APIs consumed by the mobile team
- Containerized services with Docker for local development

Projects
- Expense Tracker API: FastAPI + PostgreSQL service with JWT auth
- Internal Search Tool: added a basic keyword search endpoint over support tickets;
  briefly explored embedding-based search but the project was shelved before completion

Education
B.Tech in Computer Science, VIT Pune (2021)
"""

candidate_02 = """Daniel Osei
Machine Learning Engineer

Summary
Machine learning engineer focused on model development and experimentation.
Comfortable with Python for data work; limited exposure to backend/API development.

Skills
Python, scikit-learn, PyTorch, Pandas, NumPy, Jupyter, SQL, Git

Experience
ML Engineer, Northbeam Analytics (2022 - Present)
- Trained and evaluated classification models for customer churn prediction
- Built data pipelines in Python/Pandas to clean and featurize raw event logs
- Presented model performance results to stakeholders using Jupyter notebooks
- Used Git for notebook and script version control

Data Analyst, Riverstone Insurance (2020 - 2022)
- Built SQL queries and dashboards for underwriting analysis
- Automated recurring reports using Python scripts

Projects
- Churn Prediction Model: gradient boosting model, ~0.81 AUC
- Notebook-based sentiment classifier for support tickets using scikit-learn

Education
M.Sc. in Statistics, University of Ghana (2020)
"""

candidate_03 = """Maria Fernandez
AI / Backend Engineer

Summary
Software engineer with 4 years of experience, the last two focused on building
LLM-powered applications in production.

Skills
Python, FastAPI, REST APIs, RAG, Vector databases, LangChain, Docker, AWS, Git

Experience
AI Engineer, Solvex AI (2023 - Present)
- Designed and shipped a RAG-based support assistant using FastAPI and a vector
  database (Pinecone) to ground LLM responses in internal documentation
- Built REST APIs for document ingestion, embedding, and retrieval
- Deployed services to AWS (ECS + S3) using Docker images
- Integrated an LLM provider's API for response generation and used LangChain for
  orchestration of multi-step retrieval chains

Software Engineer, Solvex AI (2021 - 2023)
- Built internal REST APIs in FastAPI for the analytics platform
- Set up CI checks and Docker-based local dev environments
- Used Git/GitHub for all source control, including code review

Projects
- RAG Chatbot: FastAPI + Pinecone + LLM integration, deployed on AWS
- FastAPI AI Service: internal microservice exposing embedding and search endpoints

Education
B.S. in Computer Science, Universidad de Chile (2020)
"""

candidate_04 = """Jason Lee
Frontend Engineer

Summary
Frontend-focused engineer with 3 years of experience building web interfaces.
Some exposure to backend work but primarily a JavaScript/TypeScript developer.

Skills
JavaScript, TypeScript, React, Next.js, CSS, HTML, Git, basic Python

Experience
Frontend Engineer, Bright Retail Co (2022 - Present)
- Built and maintained a React/Next.js storefront used by ~50k monthly visitors
- Worked with a small internal API layer (written by the backend team) to
  fetch product and order data
- Wrote a handful of small Python scripts for one-off data cleanup tasks
- Used Git for version control and participated in code review

UI Developer, Studio Pixel (2020 - 2022)
- Implemented responsive UI components in React
- Collaborated with designers on a component library

Projects
- E-commerce Storefront: React/Next.js frontend, connected to an existing REST API
- Personal Portfolio Site: static site with a small Python script to generate pages

Education
B.A. in Interactive Media, San Jose State University (2020)
"""

candidate_05 = """Aisha Rahman
Applied AI Engineer

Summary
Applied AI engineer with 2.5 years of experience building LLM-powered features.
Strong Python and ML background; API/backend work has mostly been in Flask rather
than FastAPI.

Skills
Python, Machine Learning, LLM application development, Flask, REST APIs, Pandas,
Git, basic Docker

Experience
Applied AI Engineer, Northline Software (2023 - Present)
- Built an LLM-powered document summarization feature using Python
- Exposed the summarization feature through a small Flask REST API
- Fine-tuned prompt templates and evaluated output quality against a labeled test set
- Wrote Python data pipelines using Pandas to prepare training/eval datasets

Machine Learning Intern, Cascade Data (2022)
- Assisted in training a text classification model
- Wrote Python scripts to clean and label raw text data

Projects
- Document Summarizer: Python + Flask service that calls an LLM API and returns
  structured summaries
- Text Classifier: scikit-learn based classifier for support ticket routing

Education
B.S. in Computer Science, North South University (2022)
"""

for name, content in [
    ("candidate_01.txt", candidate_01), ("candidate_02.txt", candidate_02),
    ("candidate_03.txt", candidate_03), ("candidate_04.txt", candidate_04),
    ("candidate_05.txt", candidate_05),
]:
    with open(f"data/resumes/{name}", "w") as f:
        f.write(content)

print("✓ 5 synthetic resumes written to data/resumes/")
print(sorted(os.listdir("data/resumes")))


✓ 5 synthetic resumes written to data/resumes/
['candidate_01.txt', 'candidate_02.txt', 'candidate_03.txt', 'candidate_04.txt', 'candidate_05.txt']


## 19. Build the Recruiting Agent

In [12]:
from llm import GeminiLLM
from recruiting_agent import RecruitingAgent

llm = GeminiLLM(api_key=GEMINI_API_KEY)
agent = RecruitingAgent(llm)
print(f"✓ Agent ready. Model: {llm.model}")


✓ Agent ready. Model: gemini-3.6-flash


## 20. Run the demo

This is the whole run: analyze job → plan → parse resumes → extract profiles → match → score → validate → rank → report. Watch the trace below.

In [13]:
with open("data/job_description.txt") as f:
    job_description = f.read()

state = agent.run(job_description, "data/resumes")



🧠 Step 1 — Analyzing job requirements
✓ 6 mandatory requirements
✓ 4 preferred requirements

📋 Step 2 — Creating evaluation plan
✓ Evaluation plan created

📄 Step 3 — Parsing resumes
✓ 5 resumes loaded

🧩 Step 4 — Extracting candidate profiles
✓ 5 candidate profiles extracted

🔍 Step 5 — Matching candidates against requirements
✓ Candidate evaluations created for 5 candidates

📊 Step 6 — Calculating deterministic scores
✓ Scores calculated


## 21. Display candidate ranking

In [14]:
print("=" * 56)
print("CANDIDATE RANKING")
print("=" * 56)
for ev in state.evaluations:
    print(f"{ev['rank']}. {ev['candidate_id']} — {ev['score_breakdown']['total']}/100  (mandatory coverage: {round(ev['mandatory_coverage']*100)}%)")

print("\n" + "=" * 56)
print("SHORTLIST (recommended for human review)")
print("=" * 56)
for ev in state.shortlisted:
    print(f"{ev['rank']}. {ev['candidate_id']} — {ev['score_breakdown']['total']}/100")

print(f"\nGemini calls this run: {state.gemini_calls}")
print(f"Candidates processed: {len(state.candidate_profiles)}")
print(f"Iterations: {state.iterations}")
print(f"\n⚠ {__import__('config').DISCLAIMER}")


CANDIDATE RANKING
1. candidate_03 — 100.0/100  (mandatory coverage: 100%)
2. candidate_05 — 75.0/100  (mandatory coverage: 83%)
3. candidate_01 — 73.12/100  (mandatory coverage: 83%)
4. candidate_04 — 46.67/100  (mandatory coverage: 50%)
5. candidate_02 — 25.42/100  (mandatory coverage: 33%)

SHORTLIST (recommended for human review)
1. candidate_03 — 100.0/100
2. candidate_05 — 75.0/100
3. candidate_01 — 73.12/100

Gemini calls this run: 3
Candidates processed: 5
Iterations: 6

⚠ This system provides decision support only.
Final hiring decisions must be made by qualified human reviewers.


## 22. Display detailed evaluations

In [15]:
for ev in state.evaluations:
    print("\n" + "-" * 56)
    print(f"Candidate: {ev['candidate_id']}")
    print(f"Overall Score: {ev['score_breakdown']['total']}/100")
    print(f"Mandatory Coverage: {round(ev['mandatory_coverage']*100)}%")
    print("\nMandatory Requirements:")
    for m in ev["mandatory_matches"]:
        sym = {"MATCH": "✓", "PARTIAL_MATCH": "~", "NO_EVIDENCE": "-"}.get(m["status"], "?")
        print(f"  {sym} {m['requirement']} ({m['status']}) {m.get('evidence', '')}")
    print("\nPreferred Requirements:")
    for m in ev["preferred_matches"]:
        sym = {"MATCH": "✓", "PARTIAL_MATCH": "~", "NO_EVIDENCE": "-"}.get(m["status"], "?")
        print(f"  {sym} {m['requirement']} ({m['status']}) {m.get('evidence', '')}")
    print("\nGaps:")
    for g in ev["gaps"]:
        print(f"  - {g}")



--------------------------------------------------------
Candidate: candidate_03
Overall Score: 100.0/100
Mandatory Coverage: 100%

Mandatory Requirements:
  ✓ Python (MATCH) 4 years of software engineering experience building Python web services and AI applications
  ✓ FastAPI (MATCH) Designed and shipped microservices and RAG applications using FastAPI
  ✓ REST APIs (MATCH) Exposed embedding and search endpoints through FastAPI microservices
  ✓ LLM application development (MATCH) 2 years focused on building production LLM-powered applications and integration
  ✓ RAG (MATCH) Designed and shipped a production RAG-based support assistant
  ✓ Git (MATCH) Listed in skills profile

Preferred Requirements:
  ✓ Docker (MATCH) Deployed services to production using Docker images
  ✓ AWS (MATCH) Deployed containerized services to AWS (ECS and S3)
  ✓ Vector databases (MATCH) Utilized Pinecone vector database for RAG support assistant
  ✓ Machine learning experience (MATCH) Hands-on experience

## 23–24. Export JSON + Markdown report

In [16]:
from recruiting_agent import export_outputs

json_path, md_path = export_outputs(state)
print(f"✓ JSON written to {json_path}")
print(f"✓ Markdown report written to {md_path}")


✓ JSON written to outputs/candidate_rankings.json
✓ Markdown report written to outputs/recruiting_report.md


In [17]:
print(state.final_report_md[:2000])
print("\n... (see outputs/recruiting_report.md for the full report)")




... (see outputs/recruiting_report.md for the full report)


## 25. Unit tests

These run without any Gemini calls — pure Python logic checks.

In [20]:
import os

os.makedirs("tests", exist_ok=True)

print("✓ tests/ directory ready")

✓ tests/ directory ready


In [21]:
%%writefile tests/test_scoring.py
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
import config, scoring

def test_weights_sum_to_one():
    assert abs(sum(config.SCORING_WEIGHTS.values()) - 1.0) < 0.001

def test_skill_score_all_match():
    assert scoring.calculate_skill_score([{"status": "MATCH"}, {"status": "MATCH"}], 0.5) == 50.0

def test_total_score_never_exceeds_100():
    components = {k: 999 for k in config.SCORING_WEIGHTS}
    breakdown = scoring.calculate_total_score(components)
    assert breakdown["total"] <= 100

def test_shortlist_respects_coverage_threshold():
    evaluations = [
        {"candidate_id": "a", "score_breakdown": {"total": 95}, "mandatory_coverage": 0.4},
        {"candidate_id": "b", "score_breakdown": {"total": 80}, "mandatory_coverage": 0.8},
    ]
    shortlist = scoring.create_shortlist(evaluations, size=3, min_coverage=0.7)
    ids = [e["candidate_id"] for e in shortlist]
    assert "a" not in ids and "b" in ids

def _run_all():
    tests = [o for n, o in list(globals().items()) if n.startswith("test_")]
    passed = 0
    for t in tests:
        t()
        passed += 1
        print(f"  ✓ {t.__name__}")
    print(f"\n{passed} passed")

if __name__ == "__main__":
    _run_all()


Writing tests/test_scoring.py


In [22]:
%%writefile tests/test_parser.py
import os, sys, shutil, tempfile
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
import parser

def test_load_txt_resume():
    d = tempfile.mkdtemp()
    try:
        path = os.path.join(d, "candidate_01.txt")
        with open(path, "w") as f:
            f.write("Python, FastAPI, RAG experience.")
        resume = parser.load_resume(path)
        assert resume["candidate_id"] == "candidate_01"
    finally:
        shutil.rmtree(d)

def test_missing_file_raises():
    try:
        parser.load_resume("/tmp/does_not_exist_12345.txt")
        assert False
    except parser.ResumeLoadError:
        pass

def test_unsupported_extension_raises():
    d = tempfile.mkdtemp()
    try:
        path = os.path.join(d, "c.docx")
        with open(path, "w") as f:
            f.write("x")
        try:
            parser.load_resume(path)
            assert False
        except parser.ResumeLoadError:
            pass
    finally:
        shutil.rmtree(d)

def _run_all():
    tests = [o for n, o in list(globals().items()) if n.startswith("test_")]
    passed = 0
    for t in tests:
        t()
        passed += 1
        print(f"  ✓ {t.__name__}")
    print(f"\n{passed} passed")

if __name__ == "__main__":
    _run_all()


Writing tests/test_parser.py


In [23]:
os.makedirs("tests", exist_ok=True)
!python tests/test_scoring.py
!python tests/test_parser.py


  ✓ test_weights_sum_to_one
  ✓ test_skill_score_all_match
  ✓ test_total_score_never_exceeds_100
  ✓ test_shortlist_respects_coverage_threshold

4 passed
  ✓ test_load_txt_resume
  ✓ test_missing_file_raises
  ✓ test_unsupported_extension_raises

3 passed


## 26. Generate the remaining GitHub project files

(`requirements.txt`, `.env.example` — everything else was already written to disk by the `%%writefile` cells above.)

In [24]:
%%writefile requirements.txt
google-genai>=0.3.0
pypdf>=4.0.0


Writing requirements.txt


In [25]:
%%writefile .env.example
# Copy this file to .env if running outside Colab (not required in Colab -
# there we read from Colab Secrets instead, see README).

GEMINI_API_KEY=your_gemini_api_key_here

# Optional - override the default model
GEMINI_MODEL=gemini-2.0-flash


Writing .env.example


## 27. Generate README.md

In [26]:
readme_text = open("README_SOURCE.md").read() if os.path.exists("README_SOURCE.md") else None

# The full README is provided alongside this notebook in the repo
# (README.md at the project root). If you're running this notebook
# standalone and want to regenerate it, paste the README content here
# and write it with %%writefile README.md in a new cell.
print("See README.md in the repository root for the full project README.")


See README.md in the repository root for the full project README.


## 28. Verify project structure

In [27]:
expected = [
    "config.py", "llm.py", "parser.py", "scoring.py", "tools.py", "recruiting_agent.py",
    "requirements.txt", ".env.example",
    "data/job_description.txt",
    "data/resumes/candidate_01.txt", "data/resumes/candidate_02.txt",
    "data/resumes/candidate_03.txt", "data/resumes/candidate_04.txt", "data/resumes/candidate_05.txt",
    "tests/test_scoring.py", "tests/test_parser.py",
    "outputs/candidate_rankings.json", "outputs/recruiting_report.md",
]

missing = [p for p in expected if not os.path.exists(p)]
if missing:
    print("⚠ Missing files:")
    for m in missing:
        print(f"  - {m}")
else:
    print("✓ All expected project files are present.")


✓ All expected project files are present.


## 29. Create the ZIP archive

In [28]:
import shutil

project_dir = "04-recruiting-agent"
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/data/resumes", exist_ok=True)
os.makedirs(f"{project_dir}/outputs", exist_ok=True)
os.makedirs(f"{project_dir}/tests", exist_ok=True)

for f in ["config.py", "llm.py", "parser.py", "scoring.py", "tools.py",
          "recruiting_agent.py", "requirements.txt", ".env.example"]:
    if os.path.exists(f):
        shutil.copy(f, f"{project_dir}/{f}")

for f in ["job_description.txt"]:
    shutil.copy(f"data/{f}", f"{project_dir}/data/{f}")

for f in sorted(os.listdir("data/resumes")):
    shutil.copy(f"data/resumes/{f}", f"{project_dir}/data/resumes/{f}")

for f in ["test_scoring.py", "test_parser.py"]:
    shutil.copy(f"tests/{f}", f"{project_dir}/tests/{f}")

for f in os.listdir("outputs"):
    shutil.copy(f"outputs/{f}", f"{project_dir}/outputs/{f}")

open(f"{project_dir}/outputs/.gitkeep", "a").close()

zip_path = shutil.make_archive("agentic-recruiting-agent", "zip", root_dir=".", base_dir=project_dir)
print(f"✓ Created {zip_path}")


✓ Created /content/agentic-recruiting-agent.zip


## 30. Download the ZIP

In [29]:
from google.colab import files

files.download("agentic-recruiting-agent.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

### GitHub upload instructions

1. Unzip `agentic-recruiting-agent.zip` locally — you'll get a
   `04-recruiting-agent/` folder.
2. Add it to your portfolio repo alongside `01-research-agent/`,
   `02-coding-agent/`, `03-sales-agent/`:
   ```bash
   cp -r 04-recruiting-agent /path/to/agentic-ai-portfolio/
   cd /path/to/agentic-ai-portfolio
   git add 04-recruiting-agent
   git commit -m "Add Project 4: Agentic Recruiting Agent"
   git push
   ```
3. Never commit a real `.env` file or an API key — `.env.example` is a
   template only, and this project never hard-codes a key anywhere.
